# tabular_agent — Phase 2: Sandbox & Read-only Exploration

Phase 1 built a *library* of honest primitives (`io`, `taskspec`, `inspect`, `cv`,
`encoders`, `metrics`, `models`, `blend`, `report`, `pipeline`). Phase 2 wraps those
primitives in **agent-facing tools**. This notebook walks the first piece of that layer:

1. **the sandbox** — how agent-supplied Python is run safely (fresh subprocess + import
   whitelist + limits), and
2. **the read-only exploration tool** (`explore_dataframe`) — how the agent checks
   hypotheses about the data ("is income skewed?", "does churn differ by city?") without
   any risk of leaking the target into the model.

It's the companion to `transform_features` (the stateless `df -> df` escape-hatch), which
reuses the very same sandbox and lands next.

> Run it cell by cell. Every cell here has been smoke-tested, but it ships unexecuted on
> purpose so you can watch each step happen.

## 0. Setup — the same churn dataset as Phase 1

A churn-style binary-classification toy set: `age`, `income` (right-skewed, with a few
missing), a categorical `city` that genuinely shifts the churn rate, and the `churn`
target. We reuse it so the exploration examples below have something real to find.

In [ ]:
import json

import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 3000

city = rng.choice(["berlin", "hamburg", "munich", "cologne", "leipzig"],
                  size=n, p=[0.30, 0.25, 0.20, 0.15, 0.10])
city_effect = {"berlin": 0.8, "hamburg": -0.5, "munich": 1.2, "cologne": -1.0, "leipzig": 0.2}
age = rng.normal(40, 12, n).round(1)
income = rng.lognormal(10, 0.4, n).round(0)

logit = 0.04 * (age - 40) + np.array([city_effect[c] for c in city]) + rng.normal(0, 1.0, n)
churn = (logit > 0.3).astype(int)

df = pd.DataFrame({"age": age, "income": income, "city": city, "churn": churn})
df.loc[rng.choice(n, 15, replace=False), "income"] = np.nan

print(f"{len(df)} rows, churn rate {df['churn'].mean():.2f}")
df.head()

## 1. Why a sandbox at all?

The agent needs to run *its own* Python — both to build stateless features and to explore
the data. Running that code in the server process would be reckless three ways:

- **Crash** — a bad line takes down the whole server.
- **Runaway** — an accidental infinite loop or a 50 GB allocation hangs everything.
- **Training / state (the real one)** — if the agent can `import lightgbm` and fit inline,
  it can recreate exactly the footgun the whole design exists to prevent (recall Phase 1,
  §5.3): a hand-rolled target encoding fit on the full data leaks the target into CV, the
  metric glows, and the shipped model is quietly worse than the report claims.

So agent code runs in a **fresh Python subprocess** with an **import whitelist**. The
whitelist is the structural part: no model library is importable, so inline training is not
*discouraged* — it is *impossible*.

## 2. The sandbox mechanism (§6)

One `subprocess` per call, with:

| Mechanism | Effect |
|---|---|
| Own process | a crash in the code can't kill the server |
| `timeout` (wall-clock) | runaway / infinite loops are hard-killed |
| CPU + memory limits | a cap on compute and RAM |
| Own temp `cwd` | it starts in an empty scratch directory |
| **Import whitelist** | `pandas`/`numpy`/`scipy` + stdlib only → **no inline training** |
| Result contract | the output must serialize to a small JSON summary |

**Honest limitation:** a subprocess is an *isolation* boundary, not a hard *security* one —
the child runs with your OS rights. That's the right trade for the threat model ("your
machine, your data"): it reliably contains buggy/runaway code and structurally blocks
training. A pluggable WASM/Docker tier is the seam for stronger isolation later.

Here's the smallest possible call — the agent passes code that defines `explore(df)`:

In [ ]:
from tabular_agent.explore import explore_dataframe

code = '''
def explore(df):
    return {"rows": len(df), "cols": df.shape[1]}
'''
explore_dataframe(df, code)

## 3. The whitelist teaches, it doesn't just block

When the code imports something outside the whitelist, the error is written *for the agent*
— a model library says "use this tool instead", so the agent redirects rather than
retrying variations of a blocked call.

In [ ]:
from tabular_agent.explore import SandboxError

# Try to train inline — structurally refused.
leaky = '''
def explore(df):
    import lightgbm as lgb   # blocked
    return "trained"
'''
try:
    explore_dataframe(df, leaky)
except SandboxError as e:
    print(e)

In [ ]:
# A non-model, non-stdlib package is refused too — the whitelist is a closed set.
try:
    explore_dataframe(df, "def explore(df):\n    import requests\n    return 1")
except SandboxError as e:
    print(e)

## 4. A crash or a runaway is contained, not fatal

An exception in the agent's code surfaces as a `SandboxError` — and the very next call still
works, because the failure happened in a *separate* process that has since exited.

In [ ]:
bad = '''
def explore(df):
    raise ValueError("something went wrong in agent code")
'''
try:
    explore_dataframe(df, bad)
except SandboxError as e:
    print("caught:", e)

# server-side, nothing is poisoned — a normal call runs fine right after:
print("still working:", explore_dataframe(df, "def explore(df):\n    return len(df)"))

In [ ]:
# A runaway loop is hard-killed by the wall-clock timeout (this cell takes ~1s).
runaway = '''
def explore(df):
    while True:
        pass
'''
try:
    explore_dataframe(df, runaway, timeout=1.0)
except SandboxError as e:
    print(e)

## 5. The read-only exploration tool

`explore_dataframe` is the agent's window into the data. The contract: `code` defines
`explore(df)` and returns a **small summary** (a scalar, a dict, or a capped
Series/DataFrame). Two things it is deliberately *not*:

- it is **not `transform_features`** — that tool returns `df -> df` and its output is
  persisted as a pipeline step that *feeds the model*. `explore` returns a summary that is
  **shown to the agent and thrown away**.
- it is **not part of the profile** — this is why we did *not* bake skew/quantiles/group-by
  into `profile_dataframe`. The agent computes exactly the statistic it needs, here.

**Skewness on demand** — the stat we chose to keep *out* of the profile. A strongly
right-skewed non-negative target is the signal for `rmsle`; a skewed feature is a `log1p`
candidate for `transform_features`:

In [ ]:
skew_code = '''
def explore(df):
    return {
        "income_skew": float(df["income"].skew()),
        "age_skew": float(df["age"].skew()),
    }
'''
explore_dataframe(df, skew_code)

**A hypothesis check** — "does the churn rate differ by city?" A group-by is a
one-liner, and the answer comes back as a small dict the agent can reason over:

In [ ]:
by_city = '''
def explore(df):
    g = df.groupby("city")["churn"].agg(["mean", "size"])
    return g.round(3).to_dict("index")
'''
explore_dataframe(df, by_city)

## 6. *Why* read-only — the leakage backdoor it closes

Looking at a per-category target mean (as we just did) is perfectly fine **for forming a
hypothesis**. The danger is the next step: turning that per-category mean into a *feature*
by hand. That is manual target-encoding on the full data — exactly the CV-leakage footgun
`fit_encoding` exists to prevent (Phase 1, §5.3 / the target-encoding tutorial). If the
agent could do that through a general Python tool, the structural guarantee would leak away
"through the back door".

`explore` closes that door two ways:

1. **No persistence path.** Its result is returned and discarded; there is no route for it
   to become a column the model trains on. The *only* sanctioned way to turn a
   category→target signal into a feature stays `fit_encoding` (out-of-fold, fit per CV
   fold).
2. **Size cap.** Even the raw values are capped, so a full-length leaky column can't be
   smuggled back out and pasted in as a feature.

Watch the cap bite: returning a full-length per-row target mean (`transform('mean')`, the
manual-target-encoding shape) comes back **truncated** — you get the shape of the answer,
never a 3000-row vector to reuse.

In [ ]:
leaky_feature = '''
def explore(df):
    # the shape of a hand-rolled target encoding — a value per row
    return df.groupby("city")["churn"].transform("mean")
'''
out = explore_dataframe(df, leaky_feature, max_rows=5)
print("kind:      ", out["__kind__"])
print("n_rows:    ", out["n_rows"], "(full length)")
print("truncated: ", out["truncated"])
print("returned:  ", len(out["data"]), "values — not a usable feature vector")

## 7. Where `profile_dataframe` sits now

With exploration on demand, the profile stays a **cheap structural fingerprint** — dtype,
missingness, cardinality — and nothing distribution-shaped. That's a deliberate split:
the profile is the always-computed first look; anything richer (skew, quantiles, group
stats) is a targeted `explore` call the agent makes when a decision actually needs it.

In [ ]:
from tabular_agent.inspect import profile_dataframe

print(json.dumps(profile_dataframe(df, target="churn"), indent=2))

## What's next

- **`transform_features`** — the stateless `df -> df` escape-hatch (§5.1). Same sandbox as
  here; the difference is the output *is* persisted as an ordered pipeline step and re-run
  at `predict`. Structurally unable to learn state (no model libs), so it's leakage-safe by
  construction.
- **`fit_encoding`** as a structured tool (target / frequency encoding), fit per fold.
- Then the §18 *option*: per-prediction explainability via post-hoc grouped TreeSHAP.